# Apartment Listing Generator (SAT Model)

Upload apartment photos → the model captions each one → assemble a draft
listing description.

**Inputs:** a folder of `.jpg/.png` images + your `best_model_bundle.pth`  
**Output:** a written description ready to paste into a real estate listing

This notebook is fully self-contained — runs on Kaggle, Colab, or any local
machine with PyTorch installed.

---
## 1. Setup

In [ ]:
# Standard imports — all should be available in any PyTorch environment
import os, json, glob, sys, subprocess
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as transforms

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__, "| Device:", device)

---
## 2. Paths — edit these

- `BUNDLE_PATH` — your trained `best_model_bundle.pth`
- `PHOTOS_DIR` — folder containing the apartment photos

In [ ]:
# ── Edit these two paths ──────────────────────────────────────────────────────
BUNDLE_PATH = "best_model_bundle.pth"     # path to your trained bundle
PHOTOS_DIR  = "apartment_photos"          # folder with .jpg/.png images

# ── Output directory (will be created if it doesn't exist) ────────────────────
OUT_DIR = "listing_output"
os.makedirs(OUT_DIR, exist_ok=True)

# Sanity checks
assert os.path.isfile(BUNDLE_PATH), f"Bundle not found: {BUNDLE_PATH}"
assert os.path.isdir(PHOTOS_DIR),   f"Photos folder not found: {PHOTOS_DIR}"

# Find all images
IMG_EXTS = ("*.jpg", "*.jpeg", "*.png", "*.webp", "*.JPG", "*.JPEG", "*.PNG")
photo_paths = []
for ext in IMG_EXTS:
    photo_paths.extend(glob.glob(os.path.join(PHOTOS_DIR, ext)))
photo_paths = sorted(photo_paths)

print(f"Found {len(photo_paths)} photo(s):")
for p in photo_paths:
    print("  -", os.path.basename(p))

---
## 3. Model architecture (must match training notebook)

These classes are copies from the training notebook so this notebook
works standalone.

In [ ]:
# ── Load the bundle first to set up vocab indices ────────────────────────────
bundle      = torch.load(BUNDLE_PATH, map_location="cpu")
word_to_idx = bundle["word_to_idx"]
idx_to_word = {int(k): v for k, v in bundle["idx_to_word"].items()}
VOCAB_SIZE  = bundle["vocab_size"]
PAD_IDX     = word_to_idx["<pad>"]
START_IDX   = word_to_idx["<start>"]
END_IDX     = word_to_idx["<end>"]
UNK_IDX     = word_to_idx["<unk>"]

print(f"Loaded bundle:")
print(f"  Model      : {bundle.get('model_name','?')}")
print(f"  BLEU-4     : {bundle.get('bleu4', 0):.4f}")
print(f"  Vocabulary : {VOCAB_SIZE:,} words")
print(f"  Attention  : {bundle.get('has_attention', False)}")

In [ ]:
# ── Attention module ──────────────────────────────────────────────────────────
class Attention(nn.Module):
    def __init__(self, encoder_dim=512, decoder_dim=512, attention_dim=512):
        super().__init__()
        self.encoder_linear = nn.Linear(encoder_dim, attention_dim)
        self.decoder_linear = nn.Linear(decoder_dim, attention_dim)
        self.score_linear   = nn.Linear(attention_dim, 1)

    def forward(self, encoder_features, decoder_hidden):
        enc_part = self.encoder_linear(encoder_features)
        dec_part = self.decoder_linear(decoder_hidden).unsqueeze(1)
        energy   = self.score_linear(torch.tanh(enc_part + dec_part)).squeeze(2)
        alpha    = torch.softmax(energy, dim=1)
        context  = (encoder_features * alpha.unsqueeze(2)).sum(dim=1)
        return context, alpha


# ── Encoders ──────────────────────────────────────────────────────────────────
class Encoder(nn.Module):
    def __init__(self, encoded_dim=512, fine_tune=False):
        super().__init__()
        resnet = models.resnet50(weights=None)
        self.resnet = nn.Sequential(*list(resnet.children())[:-2])
        self.pool   = nn.AdaptiveAvgPool2d((14, 14))
        self.linear = nn.Linear(2048, encoded_dim)

    def forward(self, images):
        feat = self.pool(self.resnet(images))
        B    = feat.shape[0]
        return self.linear(feat.permute(0,2,3,1).reshape(B, 196, 2048))


class BaselineEncoder(nn.Module):
    def __init__(self, encoded_dim=512, fine_tune=False):
        super().__init__()
        resnet = models.resnet50(weights=None)
        self.resnet = nn.Sequential(*list(resnet.children())[:-1])
        self.linear = nn.Linear(2048, encoded_dim)

    def forward(self, images):
        return self.linear(self.resnet(images).view(images.size(0), -1))


class ResNet101Encoder(nn.Module):
    def __init__(self, encoded_dim=512, fine_tune=False):
        super().__init__()
        resnet = models.resnet101(weights=None)
        self.resnet = nn.Sequential(*list(resnet.children())[:-2])
        self.pool   = nn.AdaptiveAvgPool2d((14, 14))
        self.linear = nn.Linear(2048, encoded_dim)

    def forward(self, x):
        x = self.pool(self.resnet(x))
        return self.linear(x.permute(0,2,3,1).reshape(x.shape[0], 196, 2048))


class EfficientNetEncoder(nn.Module):
    def __init__(self, encoded_dim=512, fine_tune=False):
        super().__init__()
        from torchvision.models import efficientnet_b4
        eff = efficientnet_b4(weights=None)
        self.features = eff.features
        self.pool     = nn.AdaptiveAvgPool2d((14, 14))
        self.linear   = nn.Linear(1792, encoded_dim)

    def forward(self, x):
        x = self.pool(self.features(x))
        return self.linear(x.permute(0,2,3,1).reshape(x.shape[0], 196, 1792))


class ViTEncoder(nn.Module):
    def __init__(self, encoded_dim=512, fine_tune=False):
        super().__init__()
        from torchvision.models import vit_b_16
        vit = vit_b_16(weights=None)
        self.conv_proj   = vit.conv_proj
        self.class_token = vit.class_token
        self.encoder     = vit.encoder
        self.project     = nn.Linear(768, encoded_dim)

    def forward(self, images):
        B   = images.shape[0]
        x   = self.conv_proj(images).reshape(B, 768, -1).permute(0, 2, 1)
        cls = self.class_token.expand(B, -1, -1)
        x   = self.encoder(torch.cat([cls, x], dim=1))
        return self.project(x[:, 1:, :])


# ── Decoders ──────────────────────────────────────────────────────────────────
class Decoder(nn.Module):
    def __init__(self, embed_dim, decoder_dim, vocab_size,
                 encoder_dim=512, attention_dim=512, dropout=0.5):
        super().__init__()
        self.decoder_dim = decoder_dim
        self.vocab_size  = vocab_size
        self.embedding   = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.attention   = Attention(encoder_dim, decoder_dim, attention_dim)
        self.lstm_cell   = nn.LSTMCell(embed_dim + encoder_dim, decoder_dim)
        self.init_h      = nn.Linear(encoder_dim, decoder_dim)
        self.init_c      = nn.Linear(encoder_dim, decoder_dim)
        self.beta_gate   = nn.Linear(decoder_dim, encoder_dim)
        self.dropout     = nn.Dropout(dropout)
        self.fc          = nn.Linear(decoder_dim, vocab_size)

    def init_hidden_state(self, feats):
        mean = feats.mean(dim=1)
        return torch.tanh(self.init_h(mean)), torch.tanh(self.init_c(mean))

    def generate_caption(self, encoder_features, max_len=50, beam_size=3):
        h, c = self.init_hidden_state(encoder_features)
        beams, completed = [(0.0, [START_IDX], h, c, [])], []
        for _ in range(max_len):
            new_beams = []
            for score, seq, hp, cp, alps in beams:
                last = torch.tensor([seq[-1]], device=encoder_features.device)
                emb  = self.embedding(last)
                ctx, alpha = self.attention(encoder_features, hp)
                beta = torch.sigmoid(self.beta_gate(hp))
                ctx  = beta * ctx
                hn, cn = self.lstm_cell(torch.cat([emb, ctx], dim=1), (hp, cp))
                lp     = torch.log_softmax(self.fc(hn), dim=1)
                topk_s, topk_w = lp[0].topk(beam_size)
                for s, w in zip(topk_s.tolist(), topk_w.tolist()):
                    entry = (score+s, seq+[w], hn, cn, alps+[alpha.squeeze(0).cpu()])
                    (completed if w==END_IDX else new_beams).append(entry)
            if not new_beams: break
            new_beams.sort(key=lambda x: x[0], reverse=True)
            beams = new_beams[:beam_size]
        if not completed:
            completed = [(b[0],b[1],b[2],b[3],b[4]) for b in beams]
        best = max(completed, key=lambda x: x[0])
        words = [idx_to_word[w] for w in best[1][1:]
                 if w not in (END_IDX, PAD_IDX, START_IDX, UNK_IDX)
                 and idx_to_word.get(w,"<unk>") not in ("<pad>","<start>","<end>","<unk>")]
        return words, best[4]


class BaselineDecoder(nn.Module):
    def __init__(self, embed_dim, decoder_dim, vocab_size, encoder_dim=512, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.lstm_cell = nn.LSTMCell(embed_dim, decoder_dim)
        self.init_h    = nn.Linear(encoder_dim, decoder_dim)
        self.init_c    = nn.Linear(encoder_dim, decoder_dim)
        self.dropout   = nn.Dropout(dropout)
        self.fc        = nn.Linear(decoder_dim, vocab_size)

    def generate_caption(self, feat, max_len=50, beam_size=3):
        h = torch.tanh(self.init_h(feat))
        c = torch.tanh(self.init_c(feat))
        beams, completed = [(0.0, [START_IDX], h, c)], []
        for _ in range(max_len):
            new_beams = []
            for score, seq, hp, cp in beams:
                last = torch.tensor([seq[-1]], device=feat.device)
                emb  = self.embedding(last)
                hn, cn = self.lstm_cell(emb, (hp, cp))
                lp     = torch.log_softmax(self.fc(hn), dim=1)
                topk_s, topk_w = lp[0].topk(beam_size)
                for s, w in zip(topk_s.tolist(), topk_w.tolist()):
                    entry = (score+s, seq+[w], hn, cn)
                    (completed if w==END_IDX else new_beams).append(entry)
            if not new_beams: break
            new_beams.sort(key=lambda x: x[0], reverse=True)
            beams = new_beams[:beam_size]
        if not completed:
            completed = [(b[0],b[1],b[2],b[3]) for b in beams]
        best = max(completed, key=lambda x: x[0])
        words = [idx_to_word[w] for w in best[1][1:]
                 if w not in (END_IDX, PAD_IDX, START_IDX, UNK_IDX)
                 and idx_to_word.get(w,"<unk>") not in ("<pad>","<start>","<end>","<unk>")]
        return words


print("Architecture classes ready.")

---
## 4. Build the model from the bundle

In [ ]:
has_att    = bundle["has_attention"]
model_name = bundle["model_name"]

# Pick the right encoder based on what was trained
if "ViT" in model_name:
    ENC_CLS = ViTEncoder
elif "101" in model_name:
    ENC_CLS = ResNet101Encoder
elif "Efficient" in model_name or "Eff" in model_name:
    ENC_CLS = EfficientNetEncoder
elif has_att:
    ENC_CLS = Encoder
else:
    ENC_CLS = BaselineEncoder

enc = ENC_CLS(512, False).to(device)
dec = (Decoder(256, 512, VOCAB_SIZE) if has_att
       else BaselineDecoder(256, 512, VOCAB_SIZE)).to(device)

enc.load_state_dict(bundle["encoder_state"])
dec.load_state_dict(bundle["decoder_state"])
enc.eval(); dec.eval()
print(f"Loaded {model_name}  |  attention={has_att}")

---
## 5. Caption each photo

In [ ]:
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

BEAM_SIZE = 3

# Caption each photo
photo_captions = []   # list of (filename, caption_string)
with torch.no_grad():
    for path in photo_paths:
        try:
            img = Image.open(path).convert("RGB")
        except Exception as e:
            print(f"  [skip] {os.path.basename(path)}: {e}")
            continue
        tensor = eval_transform(img).unsqueeze(0).to(device)
        feat   = enc(tensor)
        if has_att:
            words, _ = dec.generate_caption(feat, beam_size=BEAM_SIZE)
        else:
            words    = dec.generate_caption(feat, beam_size=BEAM_SIZE)
        cap = " ".join(words).strip()
        photo_captions.append((os.path.basename(path), cap))
        print(f"  {os.path.basename(path):30s} → {cap}")

print(f"\nCaptioned {len(photo_captions)} photo(s)")

---
## 6. Visual overview

In [ ]:
# Display photos with their captions
n        = len(photo_captions)
n_cols   = min(3, n)
n_rows   = (n + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols*5, n_rows*4))
axes = np.array(axes).reshape(-1) if n > 1 else [axes]

for ax, (fname, cap) in zip(axes, photo_captions):
    img_path = os.path.join(PHOTOS_DIR, fname)
    ax.imshow(Image.open(img_path).convert("RGB"))
    ax.set_title(cap, fontsize=9, wrap=True)
    ax.axis("off")
for j in range(n, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "photos_with_captions.png"), dpi=120, bbox_inches="tight")
plt.show()

---
## 7. Build the listing description

We assemble the captions into a draft description by:
1. Removing duplicate captions (different photos sometimes look the same to the model)
2. Joining them with light connective wording
3. Capitalizing the first letter and adding punctuation

This is intentionally simple — no LLM, no fancy rewriting. It produces a raw
factual description that a seller can clean up before posting.

In [ ]:
def clean_caption(c: str) -> str:
    """Light cleanup: strip whitespace, remove trailing periods."""
    return c.strip().rstrip(".").strip()


def deduplicate(captions):
    """Remove exact-duplicate captions, preserving order."""
    seen = set()
    out  = []
    for c in captions:
        c = clean_caption(c)
        if c and c not in seen:
            seen.add(c)
            out.append(c)
    return out


def assemble_description(captions):
    """
    Turn a list of captions into a paragraph.
    Format: 'Caption1. Caption2. Caption3.'
    First caption is capitalized.
    """
    if not captions:
        return "No description available."
    sentences = []
    for c in captions:
        c = clean_caption(c)
        if not c:
            continue
        # Capitalize first letter
        c = c[0].upper() + c[1:] if len(c) > 1 else c.upper()
        sentences.append(c + ".")
    return " ".join(sentences)


# ── Build description ─────────────────────────────────────────────────────────
raw_captions    = [c for _, c in photo_captions]
unique_captions = deduplicate(raw_captions)
description     = assemble_description(unique_captions)

n_total  = len(raw_captions)
n_unique = len(unique_captions)

print(f"Captions: {n_total} total, {n_unique} unique after deduplication\n")
print("=" * 60)
print("LISTING DESCRIPTION")
print("=" * 60)
print(description)
print("=" * 60)

---
## 8. Save the outputs

In [ ]:
# Save plain-text description
desc_path = os.path.join(OUT_DIR, "listing_description.txt")
with open(desc_path, "w", encoding="utf-8") as f:
    f.write(description + "\n")

# Save structured JSON (description + per-photo captions)
result = {
    "model":          bundle.get("model_name", "unknown"),
    "model_bleu4":    bundle.get("bleu4", 0),
    "n_photos":       len(photo_captions),
    "n_unique_lines": len(unique_captions),
    "per_photo": [
        {"file": fname, "caption": cap}
        for fname, cap in photo_captions
    ],
    "description":    description,
}

json_path = os.path.join(OUT_DIR, "listing.json")
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(result, f, indent=2, ensure_ascii=False)

print(f"Saved:")
print(f"  {desc_path}")
print(f"  {json_path}")
print(f"  {os.path.join(OUT_DIR, 'photos_with_captions.png')}")

---
## Done

You now have:
- `listing_description.txt` — clean text ready to copy-paste into a listing
- `listing.json`            — structured output with per-photo captions
- `photos_with_captions.png` — visual summary

### Next steps (optional, for later)
- Detect missing rooms (no caption mentions "kitchen" → flag it)
- Translate the output into French for Tunisian listings
- Add a Streamlit UI so non-technical users can drag-and-drop photos